# Approximate search: how much recall does the index give up?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/14-scale/scale.ipynb)

Built from [`cookbook/book/chapters/14-scale/scale.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/14-scale/scale.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures, from the release's
# tag on GitHub. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook @ git+https://github.com/f-inverse/jammi-ai@py-v0.49.1#subdirectory=cookbook/book"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `search` (approximate, through the HNSW sidecar
(Malkov & Yashunin 2020)) · `search(exact=True)` · `[embedding.ann] search_expansion` ·
**Theory:** approximate vs exact nearest-neighbour search and the
recall-vs-speed trade (Johnson et al. 2021) · **Rail:** measurement (the
index's recall against the engine's own exact search).

Every search in the book so far went through the embedding table's **ANN
index**: a hierarchical navigable small-world graph (Malkov & Yashunin 2020) the engine
builds beside each table, which finds near neighbours by walking a graph instead
of scoring every vector. That is what keeps search fast as a corpus grows — and
it is approximate. The question a practitioner asks before trusting it is
simple: *on my embeddings, how many of the true nearest neighbours does the
index return?*

The engine answers it directly. `search(exact=True)` scores every vector and
returns the true nearest neighbours; the default `search` walks the index. The
overlap between the two is the index's **recall** (Johnson et al. 2021).

In [ ]:
import tempfile
import time
from pathlib import Path

import jammi
from jammi_cookbook import contracts, datasets, encoders, keystone, scale

SCALE = scale.current()
MODEL = encoders.text(SCALE)
K = 10
home = Path(tempfile.mkdtemp())
db = jammi.connect(f"file://{home}")
arxiv = datasets.arxiv(db, SCALE)
table = keystone.embed(db, arxiv, SCALE)
titles = db.sql(
    f"SELECT title FROM {arxiv.papers}.public.{arxiv.papers} ORDER BY paper_id LIMIT 200"
).column("title").to_pylist()
queries = [db.encode_query(model=MODEL, query=t) for t in titles]
papers = db.sql(f"SELECT COUNT(*) FROM {arxiv.papers}.public.{arxiv.papers}").column(0)[0].as_py()
print(f"{papers} papers indexed · {len(queries)} title queries · k = {K}")

## Exact vs approximate, query by query

The exact answer is the baseline. For each query, recall@k is the fraction of
the exact top-k the index also returned — order-insensitive, one number per
query, averaged.

In [ ]:
def top(db, query, **how) -> set[str]:
    hits = db.search(arxiv.papers, query=query, k=K, embedding_table=table, **how)
    return set(hits.column("_row_id").to_pylist())


exact = [top(db, q, exact=True) for q in queries]


def recall(db) -> tuple[float, float]:
    """Mean recall@k of the approximate search against `exact`, and queries per second."""
    start = time.perf_counter()
    found = [top(db, q) for q in queries]
    qps = len(queries) / (time.perf_counter() - start)
    return sum(len(f & e) / K for f, e in zip(found, exact)) / len(queries), qps


default_recall, default_qps = recall(db)
print(f"recall@{K} at the default search effort: {default_recall:.3f}")
db.close()

In [ ]:
contracts.assert_close("scale.default_recall_at_10", default_recall, tol=0.02)

## The dial — search effort against recall

HNSW search keeps a candidate list as it walks the graph; `search_expansion`
(HNSW's *ef*) is that list's width — at least `k`, since the list holds the
results. A wider list explores more of the graph: more of the true neighbours
found, more work per query. It is a **query-time**
setting: the same index, searched with a different effort. Set in the engine's
configuration, it applies to every search a session makes, so each point below
reopens the same catalog with a new `[embedding.ann] search_expansion`.

In [ ]:
curve = {}
for ef in (K, 16, 32, 64, 128):
    config = home / f"ef{ef}.toml"
    config.write_text(f"[embedding.ann]\nsearch_expansion = {ef}\n")
    with jammi.connect(f"file://{home}", config=str(config)) as tuned:
        curve[ef] = recall(tuned)
print(f"{'ef':>4}{'recall@10':>12}{'queries/s':>12}")
for ef, (r, qps) in curve.items():
    print(f"{ef:>4}{r:>12.3f}{qps:>12.0f}")

In [ ]:
recalls = [r for r, _ in curve.values()]
assert recalls == sorted(recalls), "recall does not fall as the search widens"
contracts.assert_close("scale.recall_at_ef10", curve[K][0], tol=0.05)
contracts.assert_close("scale.recall_at_ef128", curve[128][0], tol=0.02)

Recall climbs with the effort toward the exact answer. The queries-per-second
column is measured on whatever machine renders this page, so its absolute
values mean little — its shape is the cost side of the trade. The narrowest
search visibly gives up neighbours; the default effort recovers most of them.
How much any setting gives up depends on the geometry of the embeddings: the
small fixture encoder packs every similarity into a narrow band, where many
near-equal neighbours make the graph walk slip, while a trained encoder spreads
them apart. The practitioner's rule: measure recall against
`exact=True` on your own data, then pick the lowest effort that clears the bar
your application needs.

## Bridge note

> **An index is a trade, and the engine lets you measure it.** Approximate
> search walks an HNSW graph (Malkov & Yashunin 2020) instead of scoring every vector;
> `search(exact=True)` scores every vector. Their overlap is the index's recall
> on *your* embeddings (Johnson et al. 2021), and `search_expansion` is the dial
> that trades it against speed at query time — no rebuild, the same index
> searched harder.

## References

- Malkov, Yu A., Yashunin, Dmitry A. (2020) *Efficient and Robust Approximate Nearest Neighbor Search Using Hierarchical Navigable Small World Graphs* IEEE Transactions on Pattern Analysis and Machine Intelligence DOI 10.1109/TPAMI.2018.2889473; arXiv:1603.09320.
- Johnson, Jeff, Douze, Matthijs, Jégou, Hervé (2021) *Billion-Scale Similarity Search with GPUs* IEEE Transactions on Big Data DOI 10.1109/TBDATA.2019.2921572; arXiv:1702.08734.